In [2]:
import torch
import torch.nn as nn

In [3]:
import pandas as pd

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [5]:
df = pd.read_csv("output/training_data.csv")
print(len(df))

263300


In [6]:
# Filter out empty zip codes
df = df[df['ZIP CODE'] != 0.]
print(len(df))

262933


In [7]:
zip_codes = df['ZIP CODE']
zip_codes

0         10009
1         10009
2         10009
3         10009
4         10009
          ...  
263295    10033
263296    10033
263297    10033
263298    10033
263299    10033
Name: ZIP CODE, Length: 262933, dtype: int64

In [8]:
# Step 1: Map ZIP codes to unique indices
#  Note: This picks the last index for each ZIP code
zip_to_idx = {zip_code: idx for idx, zip_code in enumerate(zip_codes)}
idx_to_zip = {idx: zip_code for zip_code, idx in zip_to_idx.items()}

In [9]:
len(zip_to_idx)

69

In [10]:
len(idx_to_zip)

69

In [11]:

# Step 2: Define the embedding layer
num_zip_codes = len(zip_codes)  # Number of unique ZIP codes
embedding_dim = 4  # Choose an appropriate embedding size
embedding_layer = nn.Embedding(num_embeddings=num_zip_codes, embedding_dim=embedding_dim)


In [12]:
list(idx_to_zip.keys())

[252079,
 252629,
 250975,
 252075,
 256095,
 252291,
 256542,
 256199,
 256514,
 261169,
 255251,
 252076,
 252282,
 248980,
 249082,
 249084,
 255915,
 256148,
 252834,
 255333,
 251558,
 251599,
 251760,
 251640,
 251912,
 259783,
 259748,
 262585,
 251932,
 262902,
 262406,
 258282,
 254047,
 256537,
 256541,
 258289,
 258279,
 259653,
 257522,
 259688,
 259750,
 261172,
 262038,
 261183,
 262932,
 262925,
 38611,
 74538,
 82001,
 76911,
 78705,
 79167,
 119614,
 195250,
 83172,
 127968,
 84731,
 85472,
 85920,
 85991,
 87398,
 87462,
 88006,
 90701,
 127906,
 128041,
 129190,
 129296,
 131346]

In [13]:
# Step 3: Convert ZIP codes into tensor indices'
zip_indices = torch.Tensor([ zip_to_idx[zip_code] for zip_code in zip_codes ]).int()
# zip_indices
# print(zip_to_idx[10009])
# for zip in zip_codes:
#     print(zip_to_idx[zip])
# zip_indices = torch.tensor([zip_to_idx['10001'], zip_to_idx['10003'], zip_to_idx['10005']])


In [14]:
zip_indices

tensor([252079, 252079, 252079,  ..., 262932, 262932, 262932],
       dtype=torch.int32)

In [15]:
# Step 4: Get the embeddings
embedded_zip_codes = embedding_layer(zip_indices)
embedded_zip_codes

tensor([[ 0.7435, -0.0095, -0.5338, -1.0244],
        [ 0.7435, -0.0095, -0.5338, -1.0244],
        [ 0.7435, -0.0095, -0.5338, -1.0244],
        ...,
        [ 0.0391,  0.5102, -0.1171,  0.4232],
        [ 0.0391,  0.5102, -0.1171,  0.4232],
        [ 0.0391,  0.5102, -0.1171,  0.4232]], grad_fn=<EmbeddingBackward0>)

In [16]:
zip_code_indices = torch.tensor([0, 3, 7])  # Example ZIP code indices
embedded_vectors = embedding_layer(zip_code_indices)
print(embedded_vectors)  # Outputs a tensor of shape (3, embedding_dim)

tensor([[ 0.0820, -0.9200,  0.0325, -1.5776],
        [ 0.2425, -0.9910, -1.1769, -0.1229],
        [ 1.7314, -1.2072,  0.6188, -0.5594]], grad_fn=<EmbeddingBackward0>)


In [17]:
print("Embedding Output:")
print(embedded_zip_codes.shape)

Embedding Output:
torch.Size([262933, 4])


In [18]:
embedded_zip_codes.shape

torch.Size([262933, 4])

# Embedding nearest Match

In [19]:
# Generate embeddings (randomly initialized for demonstration)
stored_embeddings = embedding_layer.weight.detach()  # Get stored embeddings

In [20]:
stored_embeddings.shape

torch.Size([262933, 4])

In [21]:

# Example: Given an unknown embedding (simulate by selecting an existing one)
query_embedding = embedding_layer(torch.tensor(zip_to_idx[10128])) # lookup 10128
query_embedding

tensor([ 0.6815, -1.7752, -1.0693,  1.3794], grad_fn=<EmbeddingBackward0>)

In [22]:
# Compute distances between query_embedding and all stored embeddings
distances = torch.norm(stored_embeddings - query_embedding, dim=1)  # Euclidean distance
distances


tensor([3.3240, 3.4454, 3.1870,  ..., 3.7839, 2.5951, 2.7307],
       grad_fn=<LinalgVectorNormBackward0>)

In [23]:
# Find the index of the closest match
closest_idx = torch.argmin(distances).item()
closest_idx

259748

In [24]:
# Retrieve the original ZIP code
closest_zip = idx_to_zip[closest_idx]

In [25]:
closest_zip

10128

# Train a decoder

In [53]:
# Define the decoder model (a simple classifier)
class Decoder(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super(Decoder, self).__init__()
        self.fc = nn.Linear(embedding_dim, num_classes)  # Fully connected layer
        self.softmax = nn.Softmax(dim=1)  # Softmax for classification
    
    def forward(self, x):
        return self.softmax(self.fc(x))  # Predict probabilities

In [60]:
decoder = Decoder(embedding_dim, len(zip_to_idx))
decoder= decoder.to(device)
embedded_zip_codes = embedded_zip_codes.to(device)
zip_indices = zip_indices.to(device)
embedding_layer = embedding_layer.to(device)    



In [61]:
import torch.optim as optim

In [62]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()  # Classification loss
optimizer = optim.Adam(decoder.parameters(), lr=0.01)

In [63]:
list(decoder.parameters())[0].shape

torch.Size([69, 4])

In [64]:
print(f"Model structure: {decoder}\n\n")

for name, param in decoder.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: Decoder(
  (fc): Linear(in_features=4, out_features=69, bias=True)
  (softmax): Softmax(dim=1)
)


Layer: fc.weight | Size: torch.Size([69, 4]) | Values : tensor([[ 0.3027, -0.4097,  0.0928,  0.2354],
        [ 0.1509, -0.4396,  0.2601,  0.0085]], device='mps:0',
       grad_fn=<SliceBackward0>) 

Layer: fc.bias | Size: torch.Size([69]) | Values : tensor([0.1747, 0.3685], device='mps:0', grad_fn=<SliceBackward0>) 



In [65]:
embedded_zip_codes.shape

torch.Size([262933, 4])

In [66]:
print(embedded_zip_codes.shape)  # Debugging: Check the shape

torch.Size([262933, 4])


In [67]:
# Debugging: Check the shape of embedded_zip_codes
print(f"Shape of embedded_zip_codes: {embedded_zip_codes.shape}")

# Ensure the input to the decoder matches its expected input size
batch_size = embedded_zip_codes.size(0)  # Get batch size
print(batch_size)
# if len(embedded_zip_codes.shape) > 2:
#     embedded_zip_codes = embedded_zip_codes.view(batch_size, -1)  # Flatten if needed

# # Forward pass
# outputs = decoder(embedded_zip_codes)

Shape of embedded_zip_codes: torch.Size([262933, 4])
262933


In [ ]:
# Train the decoder
num_epochs = 100
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Collect all embeddings in a batch
    zip_indices = torch.tensor([zip_to_idx[zip_code] for zip_code in zip_codes]).to(device)  # Batch of indices
    print(zip_indices)
    embedded_zip_codes = embedding_layer(zip_indices)  # Shape: [batch_size, embedding_dim]
    
    # Forward pass
    outputs = decoder(embedded_zip_codes)  # Pass the batch to the decoder
    
    # Compute loss
    loss = criterion(outputs, zip_indices)
    
    # Backpropagation and optimization
    loss.backward()
    optimizer.step()

    # Print loss every 2 epochs
    if (epoch + 1) % 2 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [2/100], Loss: 0.0000
Epoch [4/100], Loss: 0.0000
Epoch [6/100], Loss: 0.0000
Epoch [8/100], Loss: 0.0000
Epoch [10/100], Loss: 0.0000
Epoch [12/100], Loss: 0.0000
Epoch [14/100], Loss: 0.0000
Epoch [16/100], Loss: 0.0000
Epoch [18/100], Loss: 0.0000
Epoch [20/100], Loss: 0.0000
Epoch [22/100], Loss: 0.0000
Epoch [24/100], Loss: 0.0000
Epoch [26/100], Loss: 0.0000
Epoch [28/100], Loss: 0.0000
Epoch [30/100], Loss: 0.0000
Epoch [32/100], Loss: 0.0000
Epoch [34/100], Loss: 0.0000
Epoch [36/100], Loss: 0.0000
Epoch [38/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [42/100], Loss: 0.0000
Epoch [44/100], Loss: 0.0000
Epoch [46/100], Loss: 0.0000
Epoch [48/100], Loss: 0.0000
Epoch [50/100], Loss: 0.0000
Epoch [52/100], Loss: 0.0000
Epoch [54/100], Loss: 0.0000
Epoch [56/100], Loss: 0.0000
Epoch [58/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [62/100], Loss: 0.0000
Epoch [64/100], Loss: 0.0000
Epoch [66/100], Loss: 0.0000
Epoch [68/100], Loss: 0.0000
Epoch [70/100], Lo

In [72]:
# Test the decoder
test_embedding = embedded_zip_codes[2].unsqueeze(0)  # Take an example embedding
print(test_embedding)
predicted_idx = torch.argmax(decoder(test_embedding)).item()
print(f"predicted_idx: {predicted_idx}")
predicted_zip = idx_to_zip[predicted_idx]

print(f"Recovered ZIP Code: {predicted_zip}")

tensor([[ 0.7435, -0.0095, -0.5338, -1.0244]], device='mps:0',
       grad_fn=<UnsqueezeBackward0>)
predicted_idx: 28


KeyError: 28